# 1834. Single-Threaded CPU

## Topic Alignment
- **Role Relevance**: Task scheduling with priorities is fundamental to job scheduling in distributed systems, ML training job orchestration, and resource management in cloud computing.
- **Scenario**: Directly applicable to designing task schedulers for ML pipelines, managing GPU job queues, batch processing systems, and priority-based execution in microservices.

## Metadata Summary
- Source: [LeetCode - Single-Threaded CPU](https://leetcode.com/problems/single-threaded-cpu/)
- Tags: `Greedy`, `Heap`, `Priority Queue`, `Sorting`, `Simulation`
- Difficulty: Medium
- Recommended Priority: High

## Problem Statement
You are given `n` tasks labeled from `0` to `n - 1` represented by a 2D integer array `tasks`, where `tasks[i] = [enqueueTimei, processingTimei]` means that the `i-th` task will be available to process at `enqueueTimei` and will take `processingTimei` to finish processing.

You have a single-threaded CPU that can process **at most one** task at a time and will act in the following way:

- If the CPU is idle and there are no available tasks to process, the CPU remains idle.
- If the CPU is idle and there are available tasks, the CPU will choose the one with the **shortest processing time**. If multiple tasks have the same shortest processing time, it will choose the task with the smallest index.
- Once a task is started, the CPU will process the entire task without stopping.
- The CPU can finish a task then start a new one instantly.

Return the order in which the CPU will process the tasks.

**Constraints:**
- `tasks.length == n`
- `1 <= n <= 10^5`
- `1 <= enqueueTimei, processingTimei <= 10^9`

## Progressive Hints
- Hint 1: Sort tasks by enqueue time to process them chronologically.
- Hint 2: Use a min heap to track available tasks, prioritized by (processing_time, index).
- Hint 3: Simulate the CPU's execution: when idle, jump to the next enqueue time; when busy, process from the heap.
- Hint 4: Keep track of the current time and add tasks to the heap as they become available.
- Hint 5: Handle the case where CPU finishes a task but no new tasks are available (jump to next enqueue time).

## Solution Overview
The optimal approach uses greedy scheduling with a min heap:

1. **Preprocessing**: Add indices to tasks and sort by enqueue time.
2. **Simulation loop**:
   - Track current time
   - Add all tasks that have enqueued by current time to a min heap (priority: processing time, then index)
   - If heap is empty and tasks remain, jump time to next task's enqueue time
   - Otherwise, process the task with minimum (processing_time, index) from heap
   - Update current time after processing
3. **Priority**: Shortest processing time first (SPT), with index as tiebreaker.

**Why greedy works**: The Shortest Processing Time (SPT) scheduling algorithm minimizes average waiting time. By always choosing the shortest available task, we optimize throughput and minimize queue buildup.

## Detailed Explanation
### Algorithm Steps:

1. **Augment tasks with indices**:
   ```python
   indexed_tasks = [(enqueue, process, i) for i, (enqueue, process) in enumerate(tasks)]
   indexed_tasks.sort()  # Sort by enqueue time
   ```

2. **Initialize**:
   - `current_time = 0`
   - `heap = []` (min heap with (processing_time, index))
   - `task_idx = 0` (pointer to next task to enqueue)
   - `result = []` (execution order)

3. **Main simulation**:
   ```python
   while task_idx < n or heap:
       # Add all tasks that have arrived by current_time
       while task_idx < n and indexed_tasks[task_idx][0] <= current_time:
           enqueue, process, idx = indexed_tasks[task_idx]
           heappush(heap, (process, idx))
           task_idx += 1
       
       if heap:
           # Process the task with shortest processing time
           process_time, idx = heappop(heap)
           result.append(idx)
           current_time += process_time
       else:
           # CPU idle: jump to next task's enqueue time
           if task_idx < n:
               current_time = indexed_tasks[task_idx][0]
   ```

4. **Return** execution order.

### Key Insights:

**Priority Queue Design**:
- Primary key: `processing_time` (shorter is better)
- Secondary key: `index` (smaller index breaks ties)
- Python's heapq naturally handles tuple comparison

**Time Management**:
- When heap is empty, don't increment time by 1; jump directly to next task's enqueue time
- This optimization is crucial for large time gaps

**Example walkthrough** for `tasks = [[1,2],[2,4],[3,2],[4,1]]`:

| Time | Available Tasks | Heap (process, idx) | Action | Result |
|------|----------------|---------------------|--------|--------|
| 0    | None           | []                  | Jump to 1 | [] |
| 1    | Task 0         | [(2,0)]             | Process 0 | [0] |
| 3    | Task 1,2       | [(4,1), (2,2)]      | Process 2 | [0,2] |
| 5    | Task 3         | [(4,1), (1,3)]      | Process 3 | [0,2,3] |
| 6    | -              | [(4,1)]             | Process 1 | [0,2,3,1] |
| 10   | -              | []                  | Done | [0,2,3,1] |

Result: [0, 2, 3, 1]

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| Greedy with heap | O(n log n) | O(n) | Optimal solution. Sort + heap operations. |
| Greedy with array scan | O(n^2) | O(n) | Find min each time instead of heap. Slower. |
| Brute force simulation | O(n * T) | O(n) | T is max time. Too slow for large T. |
| Segment tree | O(n log n) | O(n) | Overkill for this problem. |

## Reference Implementation

In [ ]:
import heapq
from typing import List


def get_order(tasks: List[List[int]]) -> List[int]:
    """
    Determine the order in which CPU processes tasks.
    
    Args:
        tasks: List of [enqueue_time, processing_time] for each task
    
    Returns:
        List of task indices in execution order
    """
    n = len(tasks)
    
    # Add original indices and sort by enqueue time
    indexed_tasks = [(tasks[i][0], tasks[i][1], i) for i in range(n)]
    indexed_tasks.sort()
    
    result = []
    heap = []  # Min heap: (processing_time, index)
    current_time = 0
    task_idx = 0  # Pointer to next task to enqueue
    
    while task_idx < n or heap:
        # Add all tasks that have arrived by current_time
        while task_idx < n and indexed_tasks[task_idx][0] <= current_time:
            enqueue_time, process_time, idx = indexed_tasks[task_idx]
            heapq.heappush(heap, (process_time, idx))
            task_idx += 1
        
        if heap:
            # Process the task with shortest processing time
            process_time, idx = heapq.heappop(heap)
            result.append(idx)
            current_time += process_time
        else:
            # CPU is idle: jump to next task's enqueue time
            if task_idx < n:
                current_time = indexed_tasks[task_idx][0]
    
    return result

## Validation

In [ ]:
cases = [
    ([[1, 2], [2, 4], [3, 2], [4, 1]], [0, 2, 3, 1]),
    ([[7, 10], [7, 12], [7, 5], [7, 4], [7, 2]], [4, 3, 2, 0, 1]),
    ([[5, 2], [7, 2], [9, 4], [6, 3], [5, 10], [1, 1]], [5, 0, 1, 3, 2, 4]),
    ([[1, 1]], [0]),
    ([[1, 10], [2, 1], [3, 1], [4, 1]], [0, 1, 2, 3]),
    ([[100, 100], [1000000000, 1000000000]], [0, 1]),
]

for tasks, expected in cases:
    result = get_order([t[:] for t in tasks])  # Deep copy
    assert result == expected, f"Failed for tasks={tasks}: got {result}, expected {expected}"

print('All tests passed for LC 1834.')

## Complexity Analysis
- **Time Complexity**: O(n log n) where n is the number of tasks
  - Sorting indexed tasks: O(n log n)
  - Each task is pushed and popped from heap once: O(n log n)
  - Overall: O(n log n)
- **Space Complexity**: O(n)
  - Indexed tasks array: O(n)
  - Heap: O(n) in worst case (all tasks enqueue at once)
  - Result array: O(n)
- **Bottleneck**: Sorting and heap operations both contribute to O(n log n).

## Edge Cases & Pitfalls
- **Single task**: Simply return [0].
- **All tasks enqueue at once**: Becomes pure SPT scheduling.
- **Tasks with same enqueue and processing time**: Use index as tiebreaker.
- **Large time gaps**: Don't increment time by 1; jump to next enqueue time.
- **Idle CPU**: When heap is empty but tasks remain, CPU waits (jumps to next enqueue).
- **Integer overflow**: With processing times up to 10^9, use appropriate data types.
- **Index preservation**: Must track original indices through sorting.
- **Heap tuple order**: (processing_time, index) ensures correct priority.

## Follow-up Variants
- What if CPU has multiple cores (k-threaded instead of single-threaded)?
- What if tasks have priorities beyond processing time (e.g., deadlines)?
- Can you handle task dependencies (some tasks must complete before others start)?
- What if tasks can be preempted (interrupted and resumed later)?
- How would you minimize total waiting time instead of just following SPT?

## Takeaways
- Shortest Processing Time (SPT) is a fundamental greedy scheduling algorithm.
- Min heap with custom priority (tuples) enables efficient greedy selection.
- Simulation problems often require careful time management: don't iterate every unit of time.
- Sorting by arrival/enqueue time is a common preprocessing step for scheduling problems.
- Always preserve original indices when sorting to return correct results.
- The greedy choice (shortest task first) is locally optimal and leads to globally optimal throughput.
- This pattern generalizes to many real-world scheduling scenarios in distributed systems.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 1353 | Maximum Number of Events That Can Be Attended | Greedy + Heap |
| 253 | Meeting Rooms II | Greedy + Heap |
| 621 | Task Scheduler | Greedy + Heap |
| 2050 | Parallel Courses III | Topological Sort + DP |
| 630 | Course Schedule III | Greedy + Heap |